<a href="https://colab.research.google.com/github/R-kinyua/BubbleSort/blob/main/CNS4107_Network_Automation_R_Kinyua.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Ryan Kinyua- 166392
!pip install --upgrade pip
!pip install requests netmiko napalm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 32.1 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 535.8/535.8 kB 14.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.4/642.4 kB 22.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 46.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 80.8 MB/s  0:00:00
  Created wheel for ncclient: filename=ncclient-0.7.0-py3-none-any.whl size=94121 sha256=75219689188048f08a0fc8e54b19685dd7a3f204f4bdfebe4d79871ec5cd4fd8
  Stored in directory: /root/.cache/pip/wheels/7a/4b

# Network Automation Lab 1: Getting started with Netmiko
In this lab, the following will be achieved


1.   Install the following Python Libraties:


*   Netmiko


*   Requests
*  Napalm

2.  Connect to a remote Cisco IOS-XE Router
3.  Retrieve the running configuration and interface status
4. Push configuration changes (Adding a loopback interface)





## Defining the device credentials

A **Cisco DevNet Always-On Sandbox**, which is a real router that is always  reachable  over the internet.

## Roles of 'ConnectHandler'


1.   Establishes SSH connection to a network device
2.   Automatically handles the "handshake" (logging in, handles the '#' or '>' prompts).

**NB** The moment you specify a device_type e.g. cisco_ios, 'ConnectHandler' knows how to:
1. Specifically navigate that specific OS
2. eneter configuration mode
3. wait for the device to finish processing a command

### 'getpass' is a standard Python module used to safeguard one's credentials.

**Problem:** Hard coding a password directly into the script (e.g., 'password = 'M@swera123'), anyone who sees the code or has access to the Github repository can steal it.
**Solution:** 'getpass' prompts the user for a password in the terminal/ console but hides the characters as they are typed

#### How 'getpass' and 'ConnectHandler' work together
The combination of the two in real-time automation scripts is both powerful and secure.
- 'getpass' : safely collects the password from the user at runtime
- 'ConnectHandler' : Takes the password and uses it to log in into the router/switch to run the command.




In [ ]:
from netmiko import ConnectHandler # The primary function within Netmiko that acts as a universal adapter across various vendors e.g Cisco, Huawei, Juniper, Nokia, ZTE
import getpass

# Device details for Cisco DevNet Sandbox

device = {
    'device_type': 'cisco_ios',
    'host': 'devnetsandboxiosxec8k.cisco.com',
    'username': 'ryan.kinyua',
    'password': '7B_ZbOc_3Cw1TFF', # Public for we are not using getpass
    'port': 22,
}



print(f" Targeting: {device['host']}")

 Targeting: devnetsandboxiosxec8k.cisco.com


### Retrieve Information (Show the commands)

Goal: Establish a connection and pull the interface status

This is the "Read" part of the automation.

In [ ]:
try:
  # Establishing the SSH connection
  print("connecting to the device....")
  net_connect = ConnectHandler(**device)

  # Run a show command
  output = net_connect.send_command('show ip int brief')

  print("\n--- Interface Status ---")
  print(output)

  # Disconnect
  net_connect.disconnect()

except Exception as e:
  print(f"Error connecting to the device: {e}")


connecting to the device....

--- Interface Status ---
Interface              IP-Address      OK? Method Status                Protocol
GigabitEthernet1       10.10.20.148    YES NVRAM  up                    up      
GigabitEthernet2       192.168.255.254 YES NVRAM  up                    up      
GigabitEthernet3       172.31.254.254  YES NVRAM  up                    up      
Loopback0              10.10.10.1      YES NVRAM  up                    up      
Loopback10             172.31.10.10    YES NVRAM  up                    up      
Loopback11             172.31.11.11    YES NVRAM  up                    up      
Loopback99             192.168.199.1   YES manual up                    up      
Loopback100            172.31.100.100  YES NVRAM  up                    up      
Loopback101            172.31.101.101  YES NVRAM  up                    up      
Loopback191            192.168.122.11  YES other  up                    up      
Loopback200            172.31.200.200  YES NVRAM  up  

#### Pushing Configurations
Automation is particularly spectacular when you want to change settings

Next, let us create a new virtual interface called 'Loopback99'

In [ ]:
# Re-establish connections to the config change

net_connect = ConnectHandler(**device)

config_commands = [
    'interface Loopback99',
    'description Configured via Python Netmiko',
    'ip address 10.10.20.48 255.255.255.255',
    'no shutdown' # Ensure the interface is administratively up
]
print("Sending configurations commands....")
output = net_connect.send_config_set(config_commands)
print(output)


#Verify the change
verification = net_connect.send_command("show ip interface brief | include Loopback99")
print("\nVerification")
print(verification)

net_connect.disconnect()

Sending configurations commands....
configure terminal
Enter configuration commands, one per line.  End with CNTL/Z.
Router(config)#interface Loopback99
Router(config-if)#description Configured via Python Netmiko
Router(config-if)#ip address 10.10.20.48 255.255.255.255
% 10.10.20.48 overlaps with GigabitEthernet1
Router(config-if)#no shutdown
Router(config-if)#end
Router#

Verification
Loopback99             192.168.199.1   YES manual up                    up      
